# SINDYc 3 · Counterfactuals and the planner (exp06, exploratory)

Notebook 2 built a SINDYc model that predicts well on average. This notebook asks whether it is
good enough **to choose**:

1. The counterfactual sets: from one state, how much does the intent change the outcome?
2. Can the model rank the six intents from a given state?
3. Is the best choice predictable at all from the team state, and does unit-level information
   help? (`cftrain`, 2,000 states x 6 intents, against `cf`, 499 states, as hold-out)
4. The planner in the real game: what did it actually choose?

Data: `cf_*` and `cftrain_*` in `data/processed/exp06_intents/` (a switching battle up to a drawn
moment, then one intent for 4 s, then advance/nearest), `planner_main.csv`
(`scripts/eval_planners.py`, 400 close battles, the same seeds for every policy).

In [ ]:
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

from sindy_wm.envs.intents import INTENTS
from sindy_wm.io import storage
from sindy_wm.models.intent_model import DT, REACH_COLS, SHAPE, STATE, control_vector, r2
from sindy_wm.paths import DATA_DIR, RAW_DIR
from sindy_wm.states.geometry import MARINE_HP, REACH

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
RAW, PROCESSED = RAW_DIR / "exp06_intents", DATA_DIR / "processed" / "exp06_intents"
LABELS = [f"{m}/{t}" for m, t in INTENTS]
HOLD = 32  # 4 s of the branch intent, in steps

## 1 · Counterfactual states

For each battle: the state at the branch moment, and the outcome under each of the six intents.
The six battles are identical up to the branch (checked below).

In [ ]:
def load_counterfactuals(prefix):
    tables = {i: pd.read_parquet(PROCESSED / f"{prefix}_{i[0]}_{i[1]}.parquet") for i in INTENTS}
    ref = tables[INTENTS[0]]
    records, starts = [], []
    for ep, g in ref.groupby(level=0):
        k = int(round(g["branch_at"].iloc[0] / DT))
        if k >= len(g) - 8 or g["alive_blue"].iloc[k] <= 0 or g["alive_red"].iloc[k] <= 0:
            continue
        rec = {"ep": ep, "k": k, "step": g.index[k][1]}
        for label, i in zip(LABELS, INTENTS, strict=True):
            h = tables[i].loc[ep]
            assert abs(h["health_blue"].iloc[k] - g["health_blue"].iloc[k]) < 1e-9
            rec[f"final {label}"] = h["health_blue"].iloc[-1] - h["health_red"].iloc[-1]
            rec[f"win {label}"] = bool(h["battle_won"].iloc[0])
        records.append(rec)
        starts.append(g.iloc[k][STATE + SHAPE + REACH_COLS])
    return pd.DataFrame(records), pd.DataFrame(starts).reset_index(drop=True)


cf, cf_start = load_counterfactuals("cf")
cft, cft_start = load_counterfactuals("cftrain")
final_cols, win_cols = [f"final {x}" for x in LABELS], [f"win {x}" for x in LABELS]
print(f"cf: {len(cf)} states, cftrain: {len(cft)} states")

summary = pd.DataFrame(index=LABELS)
for name, d in (("cf", cf), ("cftrain", cft)):
    summary[f"{name} win rate"] = d[win_cols].mean().to_numpy()
for name, d in (("cf", cf), ("cftrain", cft)):
    best = pd.Series(d[final_cols].to_numpy().argmax(1)).map(dict(enumerate(LABELS)))
    summary[f"{name} best for (share)"] = (
        best.value_counts(normalize=True).reindex(LABELS).fillna(0).to_numpy()
    )
display(summary.round(3))
for name, d in (("cf", cf), ("cftrain", cft)):
    W = d[win_cols].to_numpy()
    print(
        f"{name}: best single intent wins {W.mean(0).max():.1%}, "
        f"best of six in hindsight {W.max(1).mean():.1%}, "
        f"the intent decides the winner in {(W.max(1) != W.min(1)).mean():.1%} of states"
    )

**How chaotic is it?** advance/nearest and advance/focus differ only in which enemy each unit
shoots. withdraw/nearest and withdraw/focus are the *same* behaviour (withdrawing units don't
shoot): any difference between those two is pure sensitivity to something else in the run.

In [ ]:
d = cft
pairs = {
    "withdraw/focus - withdraw/nearest (same behaviour)": d["final withdraw/focus"]
    - d["final withdraw/nearest"],
    "advance/focus - advance/nearest": d["final advance/focus"] - d["final advance/nearest"],
    "withdraw/nearest - advance/nearest": d["final withdraw/nearest"] - d["final advance/nearest"],
    "hold/nearest - advance/nearest": d["final hold/nearest"] - d["final advance/nearest"],
}
pd.DataFrame(
    {k: [v.mean(), v.std(), (v.abs() < 1e-9).mean()] for k, v in pairs.items()},
    index=["mean", "std", "share exactly 0"],
).T.round(3)

**What the numbers say:** fill in after reading. If the withdraw/focus vs withdraw/nearest pair is
not exactly zero, the two runs are not the same game after the branch: then part of every
"intent effect" is noise of that size.

## 2 · Can the SINDYc model rank the intents?

From each `cf` state, the models from notebook 2 roll out each intent for 4 s, then
advance/nearest, for 16 s, and pick the intent with the best predicted health margin. The reach
offset at the start relaxes (tau 1 s); depth and width follow their equations.

In [ ]:
models = {name: pickle.load(open(PROCESSED / f"model_{name}.pkl", "rb")) for name in ("sparse", "rich")}


def predicted_margins(model, start, horizon=128):
    X0, S0, R0 = (start[c].to_numpy(float) for c in (STATE, SHAPE, REACH_COLS))
    out = []
    for i in INTENTS:
        U = np.repeat(control_vector("advance", "nearest")[None], horizon, 0)
        U[:HOLD] = control_vector(*i)
        Y = model.simulate(X0, S0, np.repeat(U[None], len(X0), 0), reach0=R0, shape="model")
        out.append(Y[:, -1, 0] - Y[:, -1, 1])
    return np.column_stack(out)


def choice_score(d, choice):
    T, W = d[final_cols].to_numpy(), d[win_cols].to_numpy()
    idx = np.arange(len(d))
    return {
        "win rate": W[idx, choice].mean(),
        "mean final margin": T[idx, choice].mean(),
        "regret": (T.max(1) - T[idx, choice]).mean(),
        "picks the best": (T[idx, choice] >= T.max(1) - 1e-9).mean(),
    }


T = cf[final_cols].to_numpy()
ranking = {f"always {x}": choice_score(cf, np.full(len(cf), j)) for j, x in enumerate(LABELS)}
ranking["random"] = choice_score(cf, np.random.default_rng(0).integers(0, 6, len(cf)))
ranking["hindsight best"] = choice_score(cf, T.argmax(1))
for name, model in models.items():
    Pm = predicted_margins(model, cf_start)
    rho = np.nanmean([spearmanr(Pm[n], T[n]).statistic for n in range(len(cf))])
    ranking[f"SINDYc {name}"] = choice_score(cf, Pm.argmax(1)) | {"rank correlation": rho}
pd.DataFrame(ranking).T.round(3)

**What the numbers say:** fill in. Compare the SINDYc rows with "always advance/focus", not with
random.

## 3 · Is the best choice predictable at all?

Forget the dynamics model: learn the *effect of the choice* directly. A gradient-boosted
regressor predicts the difference in final margin between two intents from the start state,
trained on `cftrain` (cross-validated) and tested on `cf`.

Two feature sets:

- **team** (Level A + geometry): health, alive, distance, depth, width, reach, health ratio
- **team + units**: also what the per-unit snapshot shows at the branch moment: how many units
  are in reach of an enemy, how many are threatened by two or more, nearest-enemy distances,
  low-health units, spread of health, weapon cooldowns.

In [ ]:
def unit_features(prefix, d, chunk=200):
    """Per-unit summaries at the branch step, from the advance/nearest run (identical up to the branch)."""
    path = RAW / f"{prefix}_advance_nearest"
    rows = []
    for part in np.array_split(d[["ep", "step"]].to_numpy(), max(1, len(d) // chunk)):
        snaps = storage.load_snapshots(path, part[:, 0].tolist())
        snaps = snaps.merge(pd.DataFrame(part, columns=["episode_idx", "step"]), on=["episode_idx", "step"])
        for ep, g in snaps[snaps["alive"]].groupby("episode_idx"):
            f = {"ep": ep}
            sides = {s: g[g["team"] == s] for s in ("blue", "red")}
            for s, o in (("blue", "red"), ("red", "blue")):
                me, op = sides[s], sides[o]
                dist = np.hypot(
                    me["x"].to_numpy()[:, None] - op["x"].to_numpy()[None],
                    me["y"].to_numpy()[:, None] - op["y"].to_numpy()[None],
                )
                near, threats = dist.min(1), (dist < REACH).sum(1)
                hp = me["hp"].to_numpy() / MARINE_HP
                f |= {
                    f"in_reach_{s}": (near < REACH).sum() / 10,
                    f"threatened2_{s}": (threats >= 2).sum() / 10,
                    f"near_min_{s}": near.min(),
                    f"near_mean_{s}": near.mean(),
                    f"near_std_{s}": near.std(),
                    f"low_hp_{s}": (hp < 0.25).sum() / 10,
                    f"hp_std_{s}": hp.std(),
                    f"hp_in_reach_{s}": hp[near < REACH].sum() / 10,
                    f"cooldown_{s}": me["cooldown"].mean(),
                }
            rows.append(f)
    return d[["ep"]].merge(pd.DataFrame(rows), on="ep", how="left").drop(columns="ep")


cf_units, cft_units = unit_features("cf", cf), unit_features("cftrain", cft)
TEAM = STATE + SHAPE + REACH_COLS + ["ratio"]
for start in (cf_start, cft_start):
    start["ratio"] = start["health_blue"] / start["health_red"]
features = {"team": TEAM, "team + units": TEAM + list(cf_units.columns)}
train_x, test_x = pd.concat([cft_start, cft_units], axis=1), pd.concat([cf_start, cf_units], axis=1)
print(f"unit features: {list(cf_units.columns)}")

In [ ]:
targets = {
    "withdraw - advance": ("withdraw/nearest", "advance/nearest"),
    "hold - advance": ("hold/nearest", "advance/nearest"),
    "focus - nearest (advance)": ("advance/focus", "advance/nearest"),
}


def diff(d, a, b):
    return (d[f"final {a}"] - d[f"final {b}"]).to_numpy()


def learner():
    return HistGradientBoostingRegressor(max_iter=200, max_depth=3, learning_rate=0.05)


pred_test = {}
table = []
for tname, (a, b) in targets.items():
    y_train, y_test = diff(cft, a, b), diff(cf, a, b)
    for fname, cols in features.items():
        cv = cross_val_predict(learner(), train_x[cols], y_train, cv=5)
        model = learner().fit(train_x[cols], y_train)
        pred_test[tname, fname] = model.predict(test_x[cols])
        table.append(
            {
                "difference": tname,
                "features": fname,
                "R2 cross-validated (cftrain)": r2(y_train, cv),
                "R2 hold-out (cf)": r2(y_test, pred_test[tname, fname]),
                "sign right (cf)": np.mean(np.sign(pred_test[tname, fname]) == np.sign(y_test)),
            }
        )
pd.DataFrame(table).set_index(["difference", "features"]).round(3)

**Turn the predictions into a choice.** Among the three sensible intents (advance/nearest,
advance/focus, withdraw/nearest), pick the one with the best predicted final margin, relative to
advance/nearest.

In [ ]:
options = ["advance/nearest", "advance/focus", "withdraw/nearest"]
col = {x: LABELS.index(x) for x in options}
for fname in features:
    pred = np.column_stack(
        [
            np.zeros(len(cf)),
            pred_test["focus - nearest (advance)", fname],
            pred_test["withdraw - advance", fname],
        ]
    )
    choice = np.array([col[x] for x in options])[pred.argmax(1)]
    ranking[f"learned from cftrain, {fname}"] = choice_score(cf, choice)
    shares = pd.Series(choice).map(dict(enumerate(LABELS))).value_counts(normalize=True).round(2).to_dict()
    print(f"{fname}: chooses {shares}")
pd.DataFrame(ranking).T.drop(columns="rank correlation").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
y = diff(cf, "withdraw/nearest", "advance/nearest")
for ax, fname in zip(axes, features, strict=True):
    ax.scatter(pred_test["withdraw - advance", fname], y, s=6, alpha=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set(xlabel="predicted", title=f"withdraw - advance, {fname}")
axes[0].set_ylabel("true difference in final margin (cf)");

**What the numbers say:** fill in. Questions to ask of the table: does adding unit features move
the hold-out R² from about 0 to clearly above 0? Is "sign right" above what always guessing the
majority sign gives?

## 4 · The planner in the real game

400 close battles, the same seeds for every policy. The model planner replans every 1 s,
commits to an intent for 4 s, and looks 12 s ahead. The oracle plans the same way with a copy of
the real game as its "model".

In [ ]:
plan = pd.read_csv(PROCESSED / "planner_main.csv")
names = {
    "fixed:hold/nearest": "hold/nearest",
    "random switching": "random switching",
    "model:rich:model:4:12": "planner, rich model",
    "fixed:advance/nearest": "advance/nearest",
    "model:sparse:hold:4:12": "planner, sparse, shape held",
    "fixed:advance/focus": "advance/focus",
    "model:sparse:model:4:12": "planner, sparse, shape model",
    "oracle": "oracle planner",
}
plan["policy"] = plan["policy"].map(names)
won = plan.pivot(index="seed", columns="policy", values="won").astype(float)[list(names.values())]
res = pd.DataFrame({"Blue wins": won.mean(), "std error": won.std() / np.sqrt(len(won))})
for ref in ("advance/focus", "advance/nearest"):
    res[f"wins where {ref} loses"] = ((won == 1) & (won[[ref]].to_numpy() == 0)).sum()
    res[f"loses where {ref} wins"] = ((won == 0) & (won[[ref]].to_numpy() == 1)).sum()
display(res.round(3))

rng = np.random.default_rng(0)
gap = won["planner, sparse, shape model"] - won["advance/focus"]
boot = [rng.choice(gap.to_numpy(), len(gap)).mean() for _ in range(5000)]
print(
    f"sparse planner - advance/focus: {gap.mean():+.3f}, paired bootstrap 95% interval "
    f"[{np.quantile(boot, 0.025):+.3f}, {np.quantile(boot, 0.975):+.3f}]"
)

**What did the planners choose?** Share of the battle time spent in each intent (0 where not
used), averaged over battles.

In [ ]:
share_cols = [c for c in plan.columns if c.startswith("share_")]
plan[share_cols] = plan[share_cols].fillna(0)
chosen = plan.groupby("policy")[share_cols].mean()
chosen.columns = [c.removeprefix("share_") for c in share_cols]
display(chosen.loc[[n for n in names.values() if n.startswith(("planner", "oracle", "random"))]].round(2))

sparse = plan[plan["policy"] == "planner, sparse, shape model"].set_index("seed")
sparse_wd = sparse["share_withdraw/nearest"] + sparse["share_withdraw/focus"]
print("battles where the sparse planner ever withdraws:", f"{(sparse_wd > 0).mean():.0%}")
print(
    "its win rate in those / the others:",
    f"{sparse.loc[sparse_wd > 0, 'won'].mean():.0%} /",
    f"{sparse.loc[sparse_wd == 0, 'won'].mean():.0%}",
    "(advance/focus on the same seeds:",
    f"{won.loc[sparse_wd[sparse_wd > 0].index, 'advance/focus'].mean():.0%} /",
    f"{won.loc[sparse_wd[sparse_wd == 0].index, 'advance/focus'].mean():.0%})",
)

**What the numbers say:** fill in.

## Questions for you

1. Section 1 measures how sensitive one run is to a change that should not matter. How large is
   that next to the intent effects? What does it mean for "best of six in hindsight"?
2. In section 3, which is it: the choice can't be predicted from *any* state you measured, or it
   can from unit-level features but not from team-level ones? What would each mean for RQ2?
3. Always hold/focus wins 70.2% and the sparse planner 67.2%, choosing advance/focus most of the
   time. Why would the model prefer advancing when holding wins more?
4. The oracle wins 94%. Is that planning, or seeing the future of a chaotic system? What
   experiment would separate the two?